# DNAStream v0.1.1-beta Tutorial

## Installation from the command line

```bash
# optional but recommended
conda create -n dnastream python=3.11
conda activate dnastream

pip install "dnastream @ git+https://github.com/VanLoo-lab/DNAStream.git"
```

Verify the installation:

```bash
python -c "import dnastream; from dnastream import DNAStream; print(dnastream.__version__)"
```

## Direct installation into the kernel
if you haven't installed into your environment/kernel already, uncomment and run the below block.

In [1]:

#!pip install "dnastream @ git+https://github.com/VanLoo-lab/DNAStream.git"

## Verify the installation 
Check that you can import the `dnsastream` package and are running the correct version (currently 0.1.1b0).

In [2]:

import dnastream; print(dnastream.__version__)


0.1.1b0


## Import packages for the tutorial

In [3]:
from pathlib import Path
import tempfile
from dnastream import DNAStream

## How to create and Stream a DNAStream file

`DNAstream.create(...)` function initializes the HDF5-file and creates all built-in datasets.  It returns a `DNAStream` object that provides an interface to the HDF5 file. 

When finished streaming the data, `close()` is used to safely close the stream in order to avoid file corruption.

In [4]:
#create a temp directory and temp file 
tmpdir = Path(tempfile.mkdtemp(prefix="dnastream_tutorial_"))
myfile = tmpdir / "temp.h5"

# Create the DNAStream file (fails if it already exists)
ds = DNAStream.create(path=str(myfile), patient_id="patientX", verbose=True)
ds.close()


2026-03-12 01:08:56 - INFO - Created DNAStream file /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_qvashm4_/temp.h5
2026-03-12 01:08:56 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_qvashm4_/temp.h5' closed.


## Opening a stream to the DNAStream file
There are two available modes for streaming a DNAStream file, 'r' and 'r+'.   
- 'r' allows read access only and prevents modifications to the DNAStream file (default)
- 'r+' allows read/write access to facilitate modfications to the DNAStream file

The `DNAStream.open(...)` function is used to open the stream to the file and returns a `DNAStream` object to interface with data.

It is **recommended** to stream a DNAStream file with a context manager. This ensure that in the event of errors or exceptions, the stream is automatically closed to print file corruption.

In [5]:
#verbose can be used to print additional messages to the console (default is False)
with DNAStream.open(path=myfile, mode="r", verbose=True) as ds:
    print(ds) #print the DNAStream object ds

2026-03-12 01:08:56 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_qvashm4_/temp.h5' open.
2026-03-12 01:08:56 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_qvashm4_/temp.h5' closed.


DNAStream for Patient patientX with Package Version: 0.1.1b0
File path: '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_qvashm4_/temp.h5'
Created by: llweber on host LM4FNLXK2L at 2026-03-12T06:08:56.412817Z
Mode: 'r'
Dataset groups:
	'canonical': 0 tables
	'links': 0 tables
	'measurements': 0 tables
	'provenance': 1 tables
	'registry': 3 tables
	'results': 0 tables



## What is included currently in a DNAStream?

`DNAStream` currently includes three built-in registries/provenance attributes and an I/O interface attribute
- `sample`  : sample registry
- `variant` : variant registry
- `snp` : SNP registry
- `log` : File modification provenance
- `io` : I/O interface for DNAStream file

They can be accessed as attributes from a `DNAStream` object or by name via the `registry(name)` or `provenance(name)` function within DNAStream.

Additionally, there are two properties:
- `patient_id` : the id of the patient under study
- `mode` : the current streaming mode of the DNAStream interface

In [6]:
with DNAStream.open(myfile) as ds:
    print(ds.sample) #a reference to the sample registry object
    print(ds.registry("sample")) # a function that returns a reference
    print(ds.log)
    print(ds.provenance("log"))

/registry/sample (Shape: (0,))
/registry/sample (Shape: (0,))
/provenance/log (Shape: (4,))
/provenance/log (Shape: (4,))


### Change or update the patient id

In [7]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Current patient id: {ds.patient_id}")
    ds.set_patient_id("patientY")
    # ds.patient_id = "foo" throws an exception
    print(f"New patient id: {ds.patient_id}")

Current patient id: patientX
New patient id: patientY


## Registry objects and their role in DNAStream
Registries play a key role in DNAStream because they provide tracking and management of key research entities, such as samples, cells, SNVs, indels, SNPs. 

When multiple analysts are working on a project, it can be difficult to track which samples pass QA and are to be used in downstream analysis or what SNVs are the current active set when multiple SNV callers are run. 

It can be hard to manage metadata and easily format it to streamline bash and workflow scripts/pipelines. They also provide link to various measured data and computed analysis results. 

Formally, a registry is a structured 1-d append only dataset (think dataframe) stored in the DNAStream HDF5 file for perpetuity. Any modifications to the registry dataset are logged so registry event histories can be extracted. 

When streaming a DNAStream file, `Registry` objects serve as interfaces to the registry dataset. Via `Registry` objects, one can:   
- `add` new entities for registration
- query the registry to `get` a subset meeting selector critera, 
- `update` the metadata of editable fields
- `activate` entities by `id` or `label`
- `find` all entities by a given `label`
- extract all or part of the registry dataset `to_dataframe`

## Add entities to the registry
There are multiple options to register entities. Registry items can be added directly to a `Registry` object via the low-level `Registry.add(...)` function from a list of dictionaries, where each dictionary is an entity.  

But there is also an `io` class to help streamline insertation from standard formats ('maf' files) and CSV and TSV files. 

### Low-level insertion
To utilize low-level insertation, first obtain a referenece to a 'Registry' object and then call the `add(...)` function on a list of dictionaries.


There are two key parameters to the `add(...)` function which deal with the fact that internal `labels` are not required to be unique:
- `allow_duplicate_labels` : whether to allow the insertation to proceed with inserting new records that collide with existng registere entities (default is False).
- `activate_newest` : in the event of duplicate labels, this specifies whether to activate the new entity upon insertation (and deactivate the old entity) or whether to leave the newly inserted colliding entity deactivated.

In [8]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    records = [  
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ] #list of dictionaries, each entity is a dictionary if field name as key

    reg.add(records,allow_duplicate_labels=True, 
            defaults={"organism": "human", "center_name": "MDACC"} )

    print(f"After add, sample registry contains {len(reg)} entities.")

Sample registry contains 0 entities.
After add, sample registry contains 2 entities.


In [9]:
with DNAStream.open(myfile, mode="r+") as ds:
       
    ds.variant.add([
        {"chrom": "chr1", "start_pos": 1231, "end_pos": 1232, "ref_allele": "A", "alt_allele": "T"},
    ])

    for snv in ds.variant:
        print(f"SNV id: {snv['id']}, SNV label: {snv['label']}, active {snv['active']}")

SNV id: 37c431d1-77cf-4392-b9e9-b292b3642967, SNV label: chr1:1231:A:T, active True


## Key Python special methods
The `Registry` interface implements special Python methods for ease of use:
- `__len__`: get the number of registered entities,
- `__iter__` : iterate through each entity in the registry
- `__get__` : slice the registry using the unique id of an entry
- `__contains__` : check if a unique id is present in the registry 

In [10]:
with DNAStream.open(path=myfile) as ds:
    print(f"Sample registry has {len(ds.sample)} entities") #get number of entities
    
    for sample in ds.sample:   #each item is a dictionary with field name as key
        print(f'label: {sample["label"]}, organism: {sample["organism"]}, sample_name {sample["sample_name"]}, created at:{sample["created_at"]}')    
        unique_id = sample["id"]
        assert unique_id in ds.sample #check if registry contains a unique id
    
    my_favorite_record = ds.sample[unique_id] #index the registry via the unique id 
for field, value in my_favorite_record.items():
    print(f"{field}: {value}")

Sample registry has 2 entities
label: S1, organism: human, sample_name S1, created at:2026-03-12T06:08:56.447757Z
label: S2, organism: human, sample_name S2, created at:2026-03-12T06:08:56.447757Z
id: 7b0075a0-9729-432f-8140-14ef2db3e7a8
label: S2
idx: 1
active: True
created_at: 2026-03-12T06:08:56.447757Z
created_by: llweber
modified_at: 2026-03-12T06:08:56.447757Z
modified_by: llweber
sample_name: S2
tissue_type: 
organism: human
library_strategy: 
library_source: 
library_selection: 
library_layout: 
read_length: 0
platform: 
model: 
center_name: MDACC
run: 
study: 
coverage: nan
modality: single-cell
location: 
bam_file_path: 
batch_id: 
reference_build: 
date_of_sequencing: 
source_file: 
source_file_id: 


## Update the metadata of registered entities
Only certain fields in a registry are modifiable. 

Registry datasets are only modifiable via the `Registry.update(...)` function.  

To provide additional guardrails, the unique id of an entity is needed to modify its metadata.  

In this example, we will look up the unique ids associated with sample 'S1' and then we will change its 'modality' in the registry to 'lcm'.  

We pass update a list of dictionaries, where each dictionary must key contain the key 'id' and any fieldnames with associated values to update.

In [11]:
#let's change the modality from bulk to lcm.
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before....")
    for entry in ds.sample:
        print(f"id: {entry['id']}, label: {entry['label']}, modality: {entry['modality']}")
    
    print("After....")
    sample_ids =ds.sample.find_ids(labels="S1")  #looks up the unique ids associated with sample "S1"
    rows = [{"id": id, "modality": "lcm"} for id in sample_ids["S1"]]
    ds.sample.update(rows, warn_missing=True)
    for entry in ds.sample:
        print(f"id: {entry['id']}, label: {entry['label']}, modality: {entry['modality']}")

Before....
id: bd68f5e0-1d99-4381-9523-1e139214b07d, label: S1, modality: bulk
id: 7b0075a0-9729-432f-8140-14ef2db3e7a8, label: S2, modality: single-cell
After....
id: bd68f5e0-1d99-4381-9523-1e139214b07d, label: S1, modality: lcm
id: 7b0075a0-9729-432f-8140-14ef2db3e7a8, label: S2, modality: single-cell


# Activate and deactivate registered entities

One of the main advantages of registering entities is that all users can view and track which the current set of research entities to include in their downstream analysis. 

`Registry` interfaces provides functionality to activate and deactive entities by 'id' or 'label'.  

**Note:** Caution should be used when activating and deactivating entities by label as there may be multiple labels. Review the DNAStream policy and pay careful attention to function arguments to ensure proper activation occurs.

Suppose sample 'S1' fails QA, lets deactivate all entities associated with sample 'S1'. 

In [12]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")
       
    ds.sample.deactivate_labels(labels="S1")

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Before...
id: bd68f5e0-1d99-4381-9523-1e139214b07d active: True
After...
id: bd68f5e0-1d99-4381-9523-1e139214b07d active: False


Now, let's reactivate sample "S1" by label. We will use the default value of `activate_newest` to ensure the most recent sample 'S1' entity added to the registry is activated.

In [13]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")
       
    ds.sample.activate_labels(labels="S1")

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Before...
id: bd68f5e0-1d99-4381-9523-1e139214b07d active: False
After...
id: bd68f5e0-1d99-4381-9523-1e139214b07d active: True


Finally, let's suppose that we want to activate the other sample 'S1' instead. We can use its unique id to ensure that entity is activated.  Any other entities with the same label will then be deactivated.

In [14]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

    id_dict = ds.sample.find_ids(labels="S1")   
    for id in id_dict["S1"]:
        if not ds.sample[id]["active"]:
            my_id = id
    ds.sample.activate_ids(ids = my_id)

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Before...
id: bd68f5e0-1d99-4381-9523-1e139214b07d active: True


NameError: name 'my_id' is not defined

## Query/export the registry or a subset of the registry to a `pandas.Dataframe`

Users can query or export views of a registry dataset as a `pandas.Dataframe` via two different functions:`to_dataframe(...)` and `get(...)` 

 **Note, these only return copies of the dataset as a dataframe and any modifications of this dataframe will not be reflected in the DNAStream HDF5 file.**


 The `to_dataframe()` function is a convenient way to extract the entire registry, or typical subsets, such as all active entities or only deactivated entities. This is passed through the `mode` parameter.  

In [ ]:
with DNAStream.open(myfile) as ds:
    sample_registry_df = ds.sample.to_dataframe(mode="all") #default
    print(f"Shape:{sample_registry_df.shape}")
    print(sample_registry_df)

    active_samples_df = ds.sample.to_dataframe(mode="active_only")
    print(f"Shape:{active_samples_df.shape}")

    non_active_samples_df = ds.sample.to_dataframe(mode="non_active")
    print(f"Shape:{non_active_samples_df.shape}")

Shape:(2, 30)
                                     id label  idx  active  \
0  c13a4cf0-76c2-470c-ad14-4264db322e21    S1    0    True   
1  fe0476cd-5aee-4c1f-88b4-57b9595db96c    S2    1    True   

                    created_at created_by                  modified_at  \
0  2026-03-12T05:19:45.952735Z    llweber  2026-03-12T05:19:45.980492Z   
1  2026-03-12T05:19:45.952735Z    llweber  2026-03-12T05:19:45.952735Z   

  modified_by sample_name tissue_type  ... study coverage     modality  \
0     llweber          S1              ...            NaN          lcm   
1     llweber          S2              ...            NaN  single-cell   

  location bam_file_path  batch_id reference_build date_of_sequencing  \
0                                                                       
1                                                                       

  source_file source_file_id  
0                             
1                             

[2 rows x 30 columns]
Shape:(2, 30)
Sha

However, users may want to make more complex queries to the registry return the records as a `pandas.Dataframe`. 

For this, the `get(...)` function should be used instead of `to_dataframe()`.  The key argument to `get(...)` is selector. `selector` can be list of labels, ids or a callable function that will subset a `pandas.Dataframe`.

In [ ]:
#subset the sample registry by entities with label "S1"
with DNAStream.open(myfile) as ds:

    S1_df = ds.sample.get(selector=["S1"], by="label")
    print(S1_df)

                                     id label  idx  active  \
0  c13a4cf0-76c2-470c-ad14-4264db322e21    S1    0    True   

                    created_at created_by                  modified_at  \
0  2026-03-12T05:19:45.952735Z    llweber  2026-03-12T05:19:45.980492Z   

  modified_by sample_name tissue_type  ... study coverage modality location  \
0     llweber          S1              ...            NaN      lcm            

  bam_file_path  batch_id reference_build date_of_sequencing source_file  \
0                                                                          

  source_file_id  
0                 

[1 rows x 30 columns]


## I/O functions
Typically, users will want to register a large number of entities from the output files of other upstream methods, like variant callers (maf files) or via CSV and TSV files.  `DNAStream` provides `io` wrappers to streamline the insertation of entities to built-in registries from files. 

Currently, `DNAStream` offers three `io` functions to add facilitate this:
1. `io.add_variants_from_maf`: add variants to the variant registry from a maf file or a list of maf files.
2. `io.add_snps_from_maf` : add SNPSs to the SNP registry from a maf file or a list of maf files.
3. `io.add_samples_from_files` : add samples to the sample registry from a CSV (default) or other delimited file.

Additionally, `DNAStream` offers static functions to parse CSV (`io.parse_csv`, `io.parse_tsv`) or TSV files for preparation in the list of dictionary format required by `Registry.add(...)`. 

First, we will generate some temporary files with example data.

In [ ]:


maf_text = "Hugo_Symbol\tChromosome\tStart_Position\tEnd_Position\tReference_Allele\tTumor_Seq_Allele2\tTumor_Sample_Barcode\nTP53\tchr17\t7579472\t7579472\tC\tT\tS1\n"
csv_text = "sample_name,modality\nS3,bulk\nS4,single-cell\n"

with tempfile.NamedTemporaryFile("w", suffix=".maf", delete=False) as maf_fp:
    maf_fp.write(maf_text)
    maf_path = maf_fp.name

with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False) as csv_fp:
    csv_fp.write(csv_text)
    csv_path = csv_fp.name

maf_path, csv_path

('/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/tmpq9t5u6h1.maf',
 '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/tmp4gu5276x.csv')

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Sample registry started with {len(ds.sample)} entities.")
    ds.io.add_samples_from_files(csv_path)
    print(f"Sample registry now contains {len(ds.sample)} entities.")
    print(ds.sample.to_dataframe())

Sample registry started with 4 entities.
Sample registry now contains 4 entities.
                                     id label  idx  active  \
0  c13a4cf0-76c2-470c-ad14-4264db322e21    S1    0    True   
1  fe0476cd-5aee-4c1f-88b4-57b9595db96c    S2    1    True   
2  9daf2bc5-b405-4956-84b1-29d45d345993    S3    2    True   
3  431063d0-622c-48e0-8537-d67681478293    S4    3    True   

                    created_at created_by                  modified_at  \
0  2026-03-12T05:19:45.952735Z    llweber  2026-03-12T05:19:45.980492Z   
1  2026-03-12T05:19:45.952735Z    llweber  2026-03-12T05:19:45.952735Z   
2  2026-03-12T05:27:49.612237Z    llweber  2026-03-12T05:27:49.612237Z   
3  2026-03-12T05:27:49.612237Z    llweber  2026-03-12T05:27:49.612237Z   

  modified_by sample_name tissue_type  ... study coverage     modality  \
0     llweber          S1              ...            NaN          lcm   
1     llweber          S2              ...            NaN  single-cell   
2     llweber 

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Variant registry started with {len(ds.variant)} entities.")
    ds.io.add_variants_from_maf(str(maf_path))
    print(f"Variant registry now contains {len(ds.variant)} entities.")
    print(ds.variant.to_dataframe())

Variant registry started with 2 entities.
Variant registry now contains 2 entities.
                                     id              label  idx  active  \
0  429db166-5ebe-4b3f-a06e-1401ad4e0a8e      chr1:1231:A:T    0    True   
1  3502483e-da43-48b8-962d-e550500c735a  chr17:7579472:C:T    1    True   

                    created_at created_by                  modified_at  \
0  2026-03-12T05:19:45.960949Z    llweber  2026-03-12T05:19:45.960949Z   
1  2026-03-12T05:28:30.112567Z    llweber  2026-03-12T05:28:30.112567Z   

  modified_by chrom  start_pos  ...  hugo entrez_gene_id  \
0     llweber  chr1       1231  ...                        
1     llweber    17    7579472  ...  TP53                  

  variant_classification variant_type dbsnp_id filter info  \
0                                                            
1                                                            

                                         source_file  source_file_id caller  
0                    

## `Provenance` objects and their role in DNAStream

 DNAStream was designed to facilitate one or more uses to complete many upstream and downstream analyses on mult-modal DNA sequencing data. 
 
 Subsequently, it becomes important to track any modifications to the DNAStream HDF5 file. Provenance datasets are used to track these modifications.  
 
 Currently, there is only one built-in provenance dataset name 'log'.  Similarily to `Registry`, DNAStream provides a specialized `Provenance` interface to a provenance dataset. 

Via `Provenance` objects, one can:   
- extract the provenance log via `to_dataframe`
- iterate through the provenance log dataset
- determine the number of logged modifications


### Provenance log fields

The provenance log contains the the following fields:
- id : unique identifier of the event (UUIDv4)
- timestamp : time and data of when the modification occurred (ISO8601 Z)
- user : the user that performed the mofidication
- scope : which interface (DNAStream, IO, Registry, etc) performed the modification
- event : type of modification; one of create, append, modify, designate
- dataset : the dataset in the HDF5 file where the modification occurred
- source : the string name of the function that was used to modify the file
- info : a JSON string that includes amplifying information about the modification


The built-in provenance modfication log can be acesssed in two ways, by attribute or by a `provenance(name)` method.

In [ ]:
with DNAStream.open(myfile) as ds:

    print(ds.log)
    log = ds.provenance("log")
    assert ds.log == log

/provenance/log (Shape: (16,))


## Using Python special methods on a `Provenance` object
The `Provenance` interface provides the following built Python special methods:
- `__iter__` : iterate through each logged modification in the provenance dataset
- `__len__` : get the total number of entries in the provenance dataset.

In [ ]:
with DNAStream.open(myfile) as ds:
    nevents = len(ds.log)
    print(f"The modification log contains {nevents} events.")

    for mod in ds.log:
        if mod["event"] == "create":
            print(mod)

The modification log contains 16 events.
{'id': '19803c9a-3d37-49b5-81bc-c18bec856ed8', 'timestamp': '2026-03-12T05:19:45.919482Z', 'user': 'llweber', 'scope': 'DNAStream', 'event': 'create', 'dataset': '/registry/sample', 'source': 'dnastream.registry.Registry.create', 'info': '{"schema_hash": "f8d138c8d856bc3807c2e4d441a0bc972bc537104aa161a482458b3efa1406ba", "schema_version": "0.1.1"}'}
{'id': '819a71a2-3372-403f-a83c-adeec2bdecc3', 'timestamp': '2026-03-12T05:19:45.920377Z', 'user': 'llweber', 'scope': 'DNAStream', 'event': 'create', 'dataset': '/registry/variant', 'source': 'dnastream.registry.Registry.create', 'info': '{"schema_hash": "8ddbdedede0b23fd80300a99d14b372278405e46318966610ef2409138ec8302", "schema_version": "0.1.1"}'}
{'id': 'ccde9270-afff-43c1-9868-b4e749374a7e', 'timestamp': '2026-03-12T05:19:45.921045Z', 'user': 'llweber', 'scope': 'DNAStream', 'event': 'create', 'dataset': '/registry/snp', 'source': 'dnastream.registry.Registry.create', 'info': '{"schema_hash": "9

## Query/export the provenance log or a subset to a `pandas.Dataframe`.
Users can export views of a proveance dataset as a `pandas.Dataframe` via  the `to_dataframe(...)` function. 

**Note, these only return copies of the dataset as a dataframe and any modifications of this dataframe will not be reflected in the DNAStream HDF5 file.** 

In [ ]:
with DNAStream.open(myfile) as ds:

    log_df = ds.log.to_dataframe()

    print(log_df)

                                      id                    timestamp  \
0   19803c9a-3d37-49b5-81bc-c18bec856ed8  2026-03-12T05:19:45.919482Z   
1   819a71a2-3372-403f-a83c-adeec2bdecc3  2026-03-12T05:19:45.920377Z   
2   ccde9270-afff-43c1-9868-b4e749374a7e  2026-03-12T05:19:45.921045Z   
3   3f1f616b-6c04-49bb-bdb1-cca724c6de9c  2026-03-12T05:19:45.923818Z   
4   acb04b18-9da8-4a44-9bb5-f3a187077ad6  2026-03-12T05:19:45.945231Z   
5   7ea4b26b-0de9-4781-8929-28678625da12  2026-03-12T05:19:45.953102Z   
6   855a7672-4d16-4e4f-8147-4e076f565f7c  2026-03-12T05:19:45.961215Z   
7   93f43993-1b3d-43cf-a538-9b587292942d  2026-03-12T05:19:45.981107Z   
8   638ca8a4-ef47-4a58-85a3-b647ad612e0b  2026-03-12T05:19:45.991257Z   
9   0fc8b673-1a1c-4693-b127-a61880461226  2026-03-12T05:19:46.002497Z   
10  b719746e-21e9-4025-807d-90b475338039  2026-03-12T05:27:49.612558Z   
11  be1e3b83-ed76-4d9f-ab12-878d3e2f3b91  2026-03-12T05:27:49.613191Z   
12  3a25b12b-b4fe-414a-afc0-0720335923bb  2026-03-1